In [1]:
import pandas as pd
from pathlib import Path

base = Path("qgis_project")
paths = {
    "aspect": base / "aspect.csv",
    "slope": base / "slope.csv",
    "vegetation": base / "vegetation.csv",
}

missing = [p for p in paths.values() if not p.exists()]
if missing:
    raise FileNotFoundError(f"Missing files in qgis_project: {', '.join(str(m) for m in missing)}")

aspect_df = pd.read_csv(paths["aspect"])
slope_df = pd.read_csv(paths["slope"])
vegetation_df = pd.read_csv(paths["vegetation"])

print(f"Loaded aspect: {len(aspect_df)} rows, slope: {len(slope_df)} rows, vegetation: {len(vegetation_df)} rows")

Loaded aspect: 43145 rows, slope: 43145 rows, vegetation: 6224 rows


In [4]:
aspect_df = aspect_df[aspect_df['value'] != -9999.0]
slope_df = slope_df[slope_df['value'] != -9999.0]
vegetation_df = vegetation_df[vegetation_df['value'] != 65535]

In [8]:
# Join aspect_df and slope_df on matching x and y coordinates
print("Before join:")
print(f"  Aspect dataframe: {len(aspect_df)} rows")
print(f"  Slope dataframe: {len(slope_df)} rows")

# Perform inner join on x and y columns
# Rename value columns to distinguish between aspect and slope values
aspect_slope_df = pd.merge(
    aspect_df.rename(columns={'value': 'aspect_value'}),
    slope_df.rename(columns={'value': 'slope_value'}),
    on=['x', 'y'],
    how='inner'
)

print(f"\nAfter join:")
print(f"  Combined dataframe: {len(aspect_slope_df)} rows")
print(f"  Columns: {aspect_slope_df.columns.tolist()}")

print("\nFirst 5 rows of joined data:")
print(aspect_slope_df.head())

Before join:
  Aspect dataframe: 18004 rows
  Slope dataframe: 18005 rows

After join:
  Combined dataframe: 18004 rows
  Columns: ['x', 'y', 'aspect_value', 'slope_value']

First 5 rows of joined data:
              x             y  aspect_value  slope_value
0 -1.316281e+07  4.048851e+06    258.420170     1.842283
1 -1.316181e+07  4.049787e+06    233.180830     9.814027
2 -1.316215e+07  4.054874e+06    203.662920    22.884270
3 -1.316322e+07  4.057144e+06     98.623566    20.707176
4 -1.316503e+07  4.054680e+06    122.746120    33.658474


In [9]:
# Join the combined aspect_slope_df with vegetation_df on matching x and y coordinates
print("Before joining with vegetation:")
print(f"  Aspect-Slope dataframe: {len(aspect_slope_df)} rows")
print(f"  Vegetation dataframe: {len(vegetation_df)} rows")

# Perform inner join with vegetation data
# Rename the vegetation value column to distinguish it
all_data_df = pd.merge(
    aspect_slope_df,
    vegetation_df.rename(columns={'value': 'vegetation_value'}),
    on=['x', 'y'],
    how='inner'
)

print(f"\nAfter joining with vegetation:")
print(f"  Final combined dataframe: {len(all_data_df)} rows")
print(f"  Columns: {all_data_df.columns.tolist()}")

print("\nFirst 5 rows of complete joined data:")
print(all_data_df.head())

print("\nData summary:")
print(f"  Aspect values range: {all_data_df['aspect_value'].min():.2f} to {all_data_df['aspect_value'].max():.2f}")
print(f"  Slope values range: {all_data_df['slope_value'].min():.2f} to {all_data_df['slope_value'].max():.2f}")
print(f"  Vegetation values range: {all_data_df['vegetation_value'].min()} to {all_data_df['vegetation_value'].max()}")

Before joining with vegetation:
  Aspect-Slope dataframe: 18004 rows
  Vegetation dataframe: 2551 rows

After joining with vegetation:
  Final combined dataframe: 0 rows
  Columns: ['x', 'y', 'aspect_value', 'slope_value', 'vegetation_value']

First 5 rows of complete joined data:
Empty DataFrame
Columns: [x, y, aspect_value, slope_value, vegetation_value]
Index: []

Data summary:
  Aspect values range: nan to nan
  Slope values range: nan to nan
  Vegetation values range: nan to nan


In [10]:
# Investigate why the join resulted in 0 matches
print("Investigating coordinate differences:")
print("\nAspect-Slope data coordinate ranges:")
print(f"  X range: {aspect_slope_df['x'].min():.2f} to {aspect_slope_df['x'].max():.2f}")
print(f"  Y range: {aspect_slope_df['y'].min():.2f} to {aspect_slope_df['y'].max():.2f}")

print("\nVegetation data coordinate ranges:")
print(f"  X range: {vegetation_df['x'].min():.2f} to {vegetation_df['x'].max():.2f}")
print(f"  Y range: {vegetation_df['y'].min():.2f} to {vegetation_df['y'].max():.2f}")

print("\nSample coordinates from aspect-slope data:")
print(aspect_slope_df[['x', 'y']].head())

print("\nSample coordinates from vegetation data:")
print(vegetation_df[['x', 'y']].head())

# Check if there are any overlapping coordinate ranges
x_overlap = (aspect_slope_df['x'].min() <= vegetation_df['x'].max()) and (vegetation_df['x'].min() <= aspect_slope_df['x'].max())
y_overlap = (aspect_slope_df['y'].min() <= vegetation_df['y'].max()) and (vegetation_df['y'].min() <= aspect_slope_df['y'].max())

print(f"\nCoordinate range overlap:")
print(f"  X ranges overlap: {x_overlap}")
print(f"  Y ranges overlap: {y_overlap}")

# Check for exact coordinate matches
print(f"\nChecking for any exact coordinate matches:")
common_coords = pd.merge(aspect_slope_df[['x', 'y']], vegetation_df[['x', 'y']], on=['x', 'y'], how='inner')
print(f"  Found {len(common_coords)} exact coordinate matches")

Investigating coordinate differences:

Aspect-Slope data coordinate ranges:
  X range: -13169898.81 to -13155976.52
  Y range: 4044847.38 to 4064730.32

Vegetation data coordinate ranges:
  X range: -13169704.16 to -13156054.16
  Y range: 4044895.08 to 4064695.08

Sample coordinates from aspect-slope data:
              x             y
0 -1.316281e+07  4.048851e+06
1 -1.316181e+07  4.049787e+06
2 -1.316215e+07  4.054874e+06
3 -1.316322e+07  4.057144e+06
4 -1.316503e+07  4.054680e+06

Sample coordinates from vegetation data:
               x            y
0  -1.316313e+07  4045915.075
2  -1.315917e+07  4054465.075
9  -1.316280e+07  4054885.075
15 -1.316661e+07  4056925.075
21 -1.315938e+07  4051975.075

Coordinate range overlap:
  X ranges overlap: True
  Y ranges overlap: True

Checking for any exact coordinate matches:
  Found 0 exact coordinate matches


In [11]:
# Try spatial join with coordinate rounding to handle precision differences
print("Attempting spatial join with coordinate rounding...")

# Round coordinates to handle precision differences
# Try different rounding levels to find matches
rounding_levels = [0, -1, -2]  # Round to integers, tens, hundreds

for round_level in rounding_levels:
    print(f"\nTrying rounding to {10**(-round_level) if round_level < 0 else 1} units...")
    
    # Create rounded versions of the dataframes
    aspect_slope_rounded = aspect_slope_df.copy()
    vegetation_rounded = vegetation_df.copy()
    
    aspect_slope_rounded['x_round'] = aspect_slope_df['x'].round(round_level)
    aspect_slope_rounded['y_round'] = aspect_slope_df['y'].round(round_level)
    
    vegetation_rounded['x_round'] = vegetation_df['x'].round(round_level)
    vegetation_rounded['y_round'] = vegetation_df['y'].round(round_level)
    
    # Try join with rounded coordinates
    test_join = pd.merge(
        aspect_slope_rounded[['x_round', 'y_round']],
        vegetation_rounded[['x_round', 'y_round']],
        on=['x_round', 'y_round'],
        how='inner'
    )
    
    print(f"  Found {len(test_join)} matches with rounding level {round_level}")
    
    if len(test_join) > 0:
        # Perform the actual join with this rounding level
        print(f"  Using rounding level {round_level} for final join...")
        
        all_data_df = pd.merge(
            aspect_slope_rounded.drop(columns=['x_round', 'y_round']),
            vegetation_rounded.rename(columns={'value': 'vegetation_value'}).drop(columns=['x_round', 'y_round']),
            left_on=[aspect_slope_rounded['x'].round(round_level), aspect_slope_rounded['y'].round(round_level)],
            right_on=[vegetation_rounded['x'].round(round_level), vegetation_rounded['y'].round(round_level)],
            how='inner'
        )
        
        print(f"  Final joined dataframe: {len(all_data_df)} rows")
        if len(all_data_df) > 0:
            print(f"  Columns: {all_data_df.columns.tolist()}")
            print("\n  First 5 rows:")
            print(all_data_df.head())
        break

Attempting spatial join with coordinate rounding...

Trying rounding to 1 units...
  Found 1 matches with rounding level 0
  Using rounding level 0 for final join...
  Final joined dataframe: 1 rows
  Columns: ['key_0', 'key_1', 'x_x', 'y_x', 'aspect_value', 'slope_value', 'x_y', 'y_y', 'vegetation_value']

  First 5 rows:
        key_0      key_1           x_x           y_x  aspect_value  \
0 -13159594.0  4049365.0 -1.315959e+07  4.049365e+06         270.0   

   slope_value           x_y          y_y  vegetation_value  
0    10.407383 -1.315959e+07  4049365.075             36026  


In [12]:
# Create a proper spatial join using coordinate rounding
print("Creating final spatial join with coordinate rounding...")

# Since rounding to integers (level 0) worked, use that approach
# Create temporary columns with rounded coordinates
aspect_slope_for_join = aspect_slope_df.copy()
vegetation_for_join = vegetation_df.copy()

aspect_slope_for_join['x_round'] = aspect_slope_df['x'].round(0)
aspect_slope_for_join['y_round'] = aspect_slope_df['y'].round(0)

vegetation_for_join['x_round'] = vegetation_df['x'].round(0)
vegetation_for_join['y_round'] = vegetation_df['y'].round(0)
vegetation_for_join = vegetation_for_join.rename(columns={'value': 'vegetation_value'})

# Perform the join using the rounded coordinates
all_data_df = pd.merge(
    aspect_slope_for_join,
    vegetation_for_join[['x_round', 'y_round', 'vegetation_value']],
    on=['x_round', 'y_round'],
    how='inner'
)

# Remove the temporary rounding columns
all_data_df = all_data_df.drop(columns=['x_round', 'y_round'])

print(f"Final spatial join results:")
print(f"  Combined dataframe: {len(all_data_df)} rows")
print(f"  Columns: {all_data_df.columns.tolist()}")

print("\nFirst 10 rows of spatially joined data:")
print(all_data_df.head(10))

if len(all_data_df) > 0:
    print("\nData summary:")
    print(f"  Aspect values range: {all_data_df['aspect_value'].min():.2f} to {all_data_df['aspect_value'].max():.2f}")
    print(f"  Slope values range: {all_data_df['slope_value'].min():.2f} to {all_data_df['slope_value'].max():.2f}")
    print(f"  Vegetation values range: {all_data_df['vegetation_value'].min()} to {all_data_df['vegetation_value'].max()}")
else:
    print("No matches found even with coordinate rounding.")

Creating final spatial join with coordinate rounding...
Final spatial join results:
  Combined dataframe: 1 rows
  Columns: ['x', 'y', 'aspect_value', 'slope_value', 'vegetation_value']

First 10 rows of spatially joined data:
              x             y  aspect_value  slope_value  vegetation_value
0 -1.315959e+07  4.049365e+06         270.0    10.407383             36026

Data summary:
  Aspect values range: 270.00 to 270.00
  Slope values range: 10.41 to 10.41
  Vegetation values range: 36026 to 36026


In [ ]:
buildings_df = pd.read_csv(Path("qgis_project") / "building_clip_processed.csv")

In [18]:
buildings_df

,fid,OBJECTID,CODE,BLD_ID,HEIGHT,ELEV,SOURCE,DATE_,STATUS,OLD_BLD_ID,AREA,Shape_Length,Shape_Area,aspect_mean,slope_mean,veg_mean,area_2,perimeter,xcoord,ycoord
0,1,1399842,Courtyard,NaN,NaN,NaN,Glendale: Merrick & Co,1992,Unchanged,NaN,101.959341,42.076875,101.959341,231.360488,0.905329,154.000000,101.959341,42.076875,6.479381e+06,1.881513e+06
1,2,1399843,Courtyard,NaN,NaN,NaN,Glendale: Merrick & Co,1992,Unchanged,NaN,235.600473,64.735237,235.600473,233.244637,3.043743,34231.266329,235.600473,64.735237,6.478275e+06,1.884290e+06
2,3,1399854,Courtyard,NaN,NaN,NaN,Glendale: LARIAC2,2008,Unchanged,NaN,2593.961633,299.095255,2593.961632,224.771942,5.966687,154.000000,2593.961632,299.095255,6.480000e+06,1.882902e+06
3,4,1399855,Courtyard,NaN,NaN,NaN,Glendale: Merrick & Co,1992,Unchanged,NaN,2847.383161,210.879243,2847.383162,265.117615,1.971596,154.000000,2847.383162,210.879243,6.479746e+06,1.877349e+06
4,5,1399860,Courtyard,NaN,NaN,NaN,Glendale: LARIAC2,2008,Unchanged,NaN,595.672069,99.907908,595.672070,178.698617,1.160151,154.000000,595.672070,99.907908,6.477956e+06,1.885221e+06
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
55261,55262,2249855,Building,202000058328,21.375901,2243.272332,LARIAC6,2020,Modified,GLEN55144,2737.096755,236.888742,2737.096754,247.097132,2.864807,1910.603478,2737.096754,236.888742,6.483039e+06,1.913235e+06
55262,55263,2249856,Building,202000058329,24.579900,2079.633051,LARIAC6,2020,Modified,GLEN54567,2986.693290,268.578860,2986.693290,264.650055,5.627166,154.000000,2986.693290,268.578860,6.481347e+06,1.913284e+06
55263,55264,2249858,Building,202000058331,18.270001,2259.767470,LARIAC6,2020,Modified,GLEN55157,3648.158179,277.558893,3648.158179,209.226501,12.918912,24267.638248,3648.158179,277.558893,6.483382e+06,1.913355e+06
55264,55265,2249859,Building,202000058332,12.509100,2338.537563,LARIAC6,2020,New,NaN,345.884860,83.507173,345.884860,168.819504,17.645345,37717.000000,345.884860,83.507173,6.484631e+06,1.913405e+06


In [19]:
# Examine the buildings dataframe structure
print("Buildings dataframe info:")
print(f"Shape: {buildings_df.shape}")
print(f"Columns: {buildings_df.columns.tolist()}")

print("\nFirst few rows:")
print(buildings_df.head())

print("\nData types:")
print(buildings_df.dtypes)

# Check for aspect, slope, and vegetation columns
aspect_cols = [col for col in buildings_df.columns if 'aspect' in col.lower()]
slope_cols = [col for col in buildings_df.columns if 'slope' in col.lower()]
veg_cols = [col for col in buildings_df.columns if 'veg' in col.lower() or 'vegetation' in col.lower()]

print(f"\nAspect-related columns: {aspect_cols}")
print(f"Slope-related columns: {slope_cols}")
print(f"Vegetation-related columns: {veg_cols}")

# Look for mean columns specifically
mean_cols = [col for col in buildings_df.columns if 'mean' in col.lower()]
print(f"Mean-related columns: {mean_cols}")

Buildings dataframe info:
Shape: (55266, 20)
Columns: ['fid', 'OBJECTID', 'CODE', 'BLD_ID', 'HEIGHT', 'ELEV', 'SOURCE', 'DATE_', 'STATUS', 'OLD_BLD_ID', 'AREA', 'Shape_Length', 'Shape_Area', 'aspect_mean', 'slope_mean', 'veg_mean', 'area_2', 'perimeter', 'xcoord', 'ycoord']

First few rows:
   fid  OBJECTID       CODE BLD_ID  HEIGHT  ELEV                  SOURCE  \
0    1   1399842  Courtyard    NaN     NaN   NaN  Glendale: Merrick & Co   
1    2   1399843  Courtyard    NaN     NaN   NaN  Glendale: Merrick & Co   
2    3   1399854  Courtyard    NaN     NaN   NaN       Glendale: LARIAC2   
3    4   1399855  Courtyard    NaN     NaN   NaN  Glendale: Merrick & Co   
4    5   1399860  Courtyard    NaN     NaN   NaN       Glendale: LARIAC2   

   DATE_     STATUS OLD_BLD_ID         AREA  Shape_Length   Shape_Area  \
0   1992  Unchanged        NaN   101.959341     42.076875   101.959341   
1   1992  Unchanged        NaN   235.600473     64.735237   235.600473   
2   2008  Unchanged        Na

In [20]:
# Analyze the distribution of each risk factor
print("Fire Risk Factor Analysis:")
print("=" * 50)

print("\nAspect Mean Statistics:")
print(f"Range: {buildings_df['aspect_mean'].min():.2f} to {buildings_df['aspect_mean'].max():.2f}")
print(f"Mean: {buildings_df['aspect_mean'].mean():.2f}")
print(f"Std: {buildings_df['aspect_mean'].std():.2f}")

print("\nSlope Mean Statistics:")
print(f"Range: {buildings_df['slope_mean'].min():.2f} to {buildings_df['slope_mean'].max():.2f}")
print(f"Mean: {buildings_df['slope_mean'].mean():.2f}")
print(f"Std: {buildings_df['slope_mean'].std():.2f}")

print("\nVegetation Mean Statistics:")
print(f"Range: {buildings_df['veg_mean'].min():.2f} to {buildings_df['veg_mean'].max():.2f}")
print(f"Mean: {buildings_df['veg_mean'].mean():.2f}")
print(f"Std: {buildings_df['veg_mean'].std():.2f}")

# Check for any missing values
print("\nMissing Values:")
print(f"Aspect: {buildings_df['aspect_mean'].isna().sum()}")
print(f"Slope: {buildings_df['slope_mean'].isna().sum()}")
print(f"Vegetation: {buildings_df['veg_mean'].isna().sum()}")

Fire Risk Factor Analysis:

Aspect Mean Statistics:
Range: 0.81 to 360.00
Mean: 203.36
Std: 53.29

Slope Mean Statistics:
Range: 0.04 to 43.24
Mean: 3.60
Std: 4.07

Vegetation Mean Statistics:
Range: 154.00 to 39174.60
Mean: 2420.97
Std: 7641.54

Missing Values:
Aspect: 17
Slope: 16
Vegetation: 50


In [21]:
import numpy as np

# Create fire risk scoring functions
def score_aspect(aspect):
    """
    Score aspect based on fire risk (1-5 scale)
    South-facing slopes (135-225 degrees) are highest risk due to sun exposure
    """
    if pd.isna(aspect):
        return 3  # Neutral score for missing data
    
    # South-facing slopes (135-225) = highest risk
    if 135 <= aspect <= 225:
        return 5
    # Southeast/Southwest (90-135, 225-270) = high risk
    elif (90 <= aspect < 135) or (225 < aspect <= 270):
        return 4
    # East/West (45-90, 270-315) = moderate risk
    elif (45 <= aspect < 90) or (270 < aspect <= 315):
        return 3
    # Northeast/Northwest (315-360/0-45) = lower risk
    else:
        return 2

def score_slope(slope):
    """
    Score slope based on fire risk (1-5 scale)
    Steeper slopes = higher fire risk due to faster fire spread
    """
    if pd.isna(slope):
        return 3  # Neutral score for missing data
    
    if slope >= 20:      # Very steep
        return 5
    elif slope >= 10:    # Steep  
        return 4
    elif slope >= 5:     # Moderate
        return 3
    elif slope >= 2:     # Gentle
        return 2
    else:                # Flat
        return 1

def score_vegetation(veg_value):
    """
    Score vegetation based on fire risk (1-5 scale)
    Higher vegetation values typically indicate denser/more flammable vegetation
    """
    if pd.isna(veg_value):
        return 3  # Neutral score for missing data
    
    # Use quantile-based scoring for vegetation
    # Higher vegetation values = higher fire risk
    if veg_value >= 10000:    # Very high vegetation
        return 5
    elif veg_value >= 5000:   # High vegetation
        return 4
    elif veg_value >= 1000:   # Moderate vegetation
        return 3
    elif veg_value >= 500:    # Low vegetation
        return 2
    else:                     # Minimal vegetation
        return 1

# Apply scoring functions
print("Calculating individual risk scores...")
buildings_df['aspect_risk'] = buildings_df['aspect_mean'].apply(score_aspect)
buildings_df['slope_risk'] = buildings_df['slope_mean'].apply(score_slope)
buildings_df['vegetation_risk'] = buildings_df['veg_mean'].apply(score_vegetation)

# Calculate composite fire risk score
# Weight: Slope (40%), Aspect (30%), Vegetation (30%)
buildings_df['fire_risk_score'] = (
    0.4 * buildings_df['slope_risk'] + 
    0.3 * buildings_df['aspect_risk'] + 
    0.3 * buildings_df['vegetation_risk']
).round(2)

# Convert composite score to 1-5 integer scale
buildings_df['fire_risk_category'] = pd.cut(
    buildings_df['fire_risk_score'], 
    bins=[0, 1.5, 2.5, 3.5, 4.5, 5.0], 
    labels=[1, 2, 3, 4, 5],
    include_lowest=True
).astype(int)

print("Fire risk scoring complete!")
print(f"\nRisk category distribution:")
print(buildings_df['fire_risk_category'].value_counts().sort_index())

Calculating individual risk scores...
Fire risk scoring complete!

Risk category distribution:
fire_risk_category
1      266
2    30299
3    20579
4     3555
5      567
Name: count, dtype: int64


In [ ]:
# Import libraries for unsupervised learning
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, calinski_harabasz_score
import matplotlib.pyplot as plt
import seaborn as sns

# Prepare the data for unsupervised learning
print("Preparing data for unsupervised learning...")

# Select features for clustering (excluding missing values)
feature_cols = ['aspect_mean', 'slope_mean', 'veg_mean']
cluster_data = buildings_df[feature_cols].dropna()

print(f"Original dataset: {len(buildings_df)} buildings")
print(f"Complete cases for clustering: {len(cluster_data)} buildings")
print(f"Features used: {feature_cols}")

# Display basic statistics
print("\nFeature statistics for clustering dataset:")
print(cluster_data.describe())